# Imports and Functions

In [1]:
import numpy as np
import pickle
import os
import pandas as pd

In [2]:
def compute_results(concatenated_matrix):
    factors_num = concatenated_matrix.shape[0]
    samples_num = concatenated_matrix.shape[1]
    scenarios_num = concatenated_matrix.shape[2]
    S = np.zeros((factors_num, scenarios_num))  # Matrix to store sensitivity results

    for factor_index in range(factors_num):
        result = concatenated_matrix[factor_index , :]
        # Compute U and UT
        U = np.sum(result[:, 0] * result[:, 1]) / samples_num
        UT = np.sum(result[:, 0] * result[:, 2]) / samples_num
        # Compute the mean outflow (F0)
        F0 = np.mean(result[:, 0])
        # Compute the variance (V)
        V = np.sum((result[:, 0] - F0) ** 2) / samples_num

        S[factor_index, 0] = 1 - (UT - F0 ** 2) / V  # Total effect of factor
        S[factor_index, 1] = (U - F0 ** 2) / V       # Main effect of factor
        S[factor_index, 2] = S[factor_index, 0] - S[factor_index, 1]
    
    return S


# Load-> Concat-> Compute

In [3]:
# DEFINE SA_reference_param and filename
SA_reference_param = 'Peak Discharge'
EVENTS_AND_BIAS_L = [('20120113', 1.56),
    ('20160108', 1.24),
    ('20180101', 1.29),
    ('20191213', 0.79)]

event_idx = 0
filename_date = EVENTS_AND_BIAS_L[event_idx][0]

date = '_'.join([filename_date[:4], filename_date[4:6], filename_date[6:]])
sim_dir = r"D:\Development\RESEARCH\Raanana\SWMM\from_radar\Sensitivity_Analysis/"
sim_dir = os.path.join(sim_dir, date, 'pickles' + '/')

In [4]:
# Initialize a variable to store concatenated sensitivity matrices
concatenated_matrices = None

# Iterate over each pickle file in the directory
for file_name in os.listdir(sim_dir):
    if file_name.startswith("VB_with_rain_factor_") and file_name.endswith(".pickle"):
        # Extract the number from the filename
        try:
            number = int(file_name.split("_")[-1].split(".")[0])
        except ValueError:
            print(f"Warning: Couldn't extract number from filename: {file_name}")
            continue

        # Check if the number is within the desired range
        if 0 <= number <= 99999:
            # Load sensitivity results from the pickle file
            with open(os.path.join(sim_dir, file_name), 'rb') as f:
                sensitivity_results = pickle.load(f)
                sensitivity_results = np.array(sensitivity_results)  # Convert to NumPy array
                # Concatenate along axis 1
                if concatenated_matrices is None:  # If it's the first iteration
                    concatenated_matrices = sensitivity_results
                else:
                    concatenated_matrices = np.concatenate((concatenated_matrices, sensitivity_results), axis=1)


In [5]:
concatenated_matrices.shape

AttributeError: 'NoneType' object has no attribute 'shape'

In [ ]:
S = compute_results(concatenated_matrices)
column_names = ['Total effect', 'Main effect', 'Interactions importance']
index_names = ['IMP', 'Storage', 'n', 'PCT_Zero', 'CN', 'Rain factor']
sensitivity_results_df = pd.DataFrame(S, columns=column_names, index=index_names)

pickle_path = os.path.join(sim_dir, 'VB_with_coxbox_final_results.pkl')

# Save the DataFrame as a pickle file
sensitivity_results_df.to_pickle(pickle_path)

bold_txt = f"\033[1m{date} and {SA_reference_param}:\033[0m"
print(f"Sensitivity Analysis Results for {bold_txt}")
sensitivity_results_df